# 📡 RevertIQ — Signal Generation

This notebook demonstrates:
1. Market regime detection and filtering
2. Multi-condition BUY/SELL signal generation
3. Signal frequency and quality analysis
4. Stock ranking and daily watchlist

In [ ]:
import warnings; warnings.filterwarnings('ignore')

from revertiq.config.settings import get_default_config
from revertiq.data.downloader import DataDownloader
from revertiq.data.cleaner import DataCleaner
from revertiq.indicators.technical import TechnicalIndicators
from revertiq.features.engineering import FeatureEngineer
from revertiq.signals.regime import RegimeFilter
from revertiq.signals.generator import SignalGenerator
from revertiq.ranking.ranker import StockRanker
from revertiq.visualization.charts import ChartEngine

import pandas as pd
import numpy as np
import plotly.express as px

config = get_default_config()
ce = ChartEngine()
print("Ready ✓")

## 1. Load & Prepare Data

In [ ]:
# Load processed data
cleaner = DataCleaner(config.data)
try:
    clean_data = cleaner.load_all_processed()
except FileNotFoundError:
    print("Downloading data first...")
    dl = DataDownloader(config.data)
    dl.download_all()
    clean_data = cleaner.clean_all()

combined = cleaner.get_combined_df()

# Get index data
dl = DataDownloader(config.data)
index_data = dl.download_index_data()
nifty_data = index_data.get('nifty50', pd.DataFrame())
if not nifty_data.empty:
    nifty_data.columns = [c.lower() for c in nifty_data.columns]
vix_data = index_data.get('india_vix', None)

# Compute indicators
ti = TechnicalIndicators()
enriched = []
for t in combined['ticker'].unique():
    sdf = combined[combined['ticker'] == t].copy().sort_values('date')
    enriched.append(ti.compute_all(sdf, config.indicator))
stock_data = pd.concat(enriched, ignore_index=True)

# Compute features
fe = FeatureEngineer(config.indicator, config.ranking)
features_df = fe.compute_all_features(stock_data, nifty_data)
print(f"Data ready: {features_df.shape}")

## 2. Market Regime

In [ ]:
rf = RegimeFilter(config.regime)

nifty_idx = nifty_data.copy()
if 'date' in nifty_idx.columns:
    nifty_idx = nifty_idx.set_index('date')

if vix_data is not None and not vix_data.empty:
    vix_data.columns = [c.lower() for c in vix_data.columns]

regime_series = rf.compute_regime(nifty_idx, vix_data)

# Regime distribution
counts = regime_series.value_counts()
fig = px.pie(values=counts.values, names=counts.index,
             title='Market Regime Distribution',
             color_discrete_map={
                 'strong_bull': '#3fb950', 'mild_bull': '#56d364',
                 'cautious': '#d29922', 'bearish': '#f85149'
             })
fig.update_layout(template='plotly_dark', paper_bgcolor='#161b22')
fig.show()

In [ ]:
# Regime chart with NIFTY price
fig = ce.regime_chart(nifty_idx.reset_index(), regime_series)
fig.show()

## 3. Signal Generation

In [ ]:
sg = SignalGenerator(config.signal)
signals_df = sg.generate_all_signals(features_df, regime_series)

n_buy = signals_df['buy_signal'].sum()
n_sell = signals_df['sell_signal'].sum()
total = len(signals_df)

print(f"Total rows    : {total:,d}")
print(f"BUY signals   : {n_buy:,d} ({n_buy/total*100:.2f}%)")
print(f"SELL signals  : {n_sell:,d} ({n_sell/total*100:.2f}%)")

In [ ]:
# Signal frequency over time
buy_freq = signals_df[signals_df['buy_signal']].groupby('date').size()
fig = px.bar(x=buy_freq.index, y=buy_freq.values,
             title='Daily BUY Signal Count Over Time',
             labels={'x': 'Date', 'y': 'Signals'})
fig.update_traces(marker_color='#3fb950')
fig.update_layout(template='plotly_dark',
                  paper_bgcolor='#161b22', plot_bgcolor='#0d1117')
fig.show()

In [ ]:
# Signal strength distribution
buy_signals = signals_df[signals_df['buy_signal']]
if not buy_signals.empty and 'signal_strength' in buy_signals.columns:
    fig = px.histogram(buy_signals, x='signal_strength', nbins=30,
                       title='Signal Strength Distribution (BUY signals)',
                       color_discrete_sequence=['#58a6ff'])
    fig.update_layout(template='plotly_dark',
                      paper_bgcolor='#161b22', plot_bgcolor='#0d1117')
    fig.show()

In [ ]:
# Most frequently signalled stocks
if not buy_signals.empty:
    top_signalled = buy_signals['ticker'].value_counts().head(15)
    fig = px.bar(x=top_signalled.values, y=top_signalled.index,
                 orientation='h', title='Most Frequently Signalled Stocks',
                 labels={'x': 'Signal Count', 'y': 'Ticker'})
    fig.update_traces(marker_color='#bc8cff')
    fig.update_layout(template='plotly_dark', yaxis=dict(autorange='reversed'),
                      paper_bgcolor='#161b22', plot_bgcolor='#0d1117')
    fig.show()

## 4. Stock Ranking

In [ ]:
ranker = StockRanker(config.ranking)
rankings = ranker.rank_all_dates(features_df, signals_df)

print(f"Ranked entries: {len(rankings):,d}")
print(f"Dates with candidates: {rankings['date'].nunique()}")

# Latest top candidates
top = ranker.get_top_candidates(rankings, n=10)
if not top.empty:
    print(f"\n--- Top Candidates ({top['date'].iloc[0].date()}) ---")
    display_cols = [c for c in ['rank','ticker','composite_score','signal_strength'] if c in top.columns]
    print(top[display_cols].to_string(index=False))

In [ ]:
# Visual ranked stocks table
fig = ce.ranked_stocks_table(rankings)
fig.show()